> **Superseded for Phase 7 (18 October 2026).** Every Phase 7 step, with its exact code, results and interpretation, now lives in `phase7-adaptive-pruning.ipynb`. This notebook is kept as the record of the Modal recovery and the Phase 6 conclusion; the launcher scripts it mentions were folded into the Phase 7 notebook.


# Phase 6/7 recovered continuation

Reconstructed on 17 October 2026 from durable Modal requests and summaries. This is not the missing original notebook. No output in this notebook is presented as a new experiment.

The original worker sources are preserved verbatim under `data/recovered/sources`; the complete small remote records are mirrored under `data/recovered/modal`. Published source hash reconciliation distinguishes recovered files from missing local-only reviews.

In [ ]:
from pathlib import Path
import json, hashlib
root = Path.cwd()
assert (root / 'PLAN.md').exists(), 'Run from the repository root'
report = json.loads((root / 'data/published/2026-10-04/reconciliation.json').read_bytes())
report['counts']


## Phase 6 conclusion

The selected artifact is the pretrained block-5 bridge. The published blinded 129-prompt review did not promote update 120. Full raw experiment requests and summaries were recovered; original local review files are listed separately as missing.

In [ ]:
published = json.loads((root / 'data/published/2026-10-04/results.json').read_bytes())
published['phase6_checkpoint_decision']['decision']


## Phase 7 candidate A

Blocks 4 and 5 are replaced by one rank-512 region bridge, followed by 60 healing updates. The 6.52% historical velocity gate failed. Candidate A remains provisional pending fresh blinded quality review.

In [ ]:
run_key = '95fcd2cfae35cb7f124b8af2865c30f928cbd681a0d015df5e56e8ae5ba4fd78'
run = root / 'data/recovered/modal/phase7/blocks45_healing' / run_key
request = json.loads((run / 'request.json').read_bytes())['request']
summary = json.loads((run / 'summary.json').read_bytes())
{key: summary[key] for key in ('status', 'selected_step', 'gate_passed_steps', 'surviving_blocks')}


In [ ]:
source = request['worker_source']
source_hash = hashlib.sha256(source.encode()).hexdigest()
assert (root / 'data/recovered/sources' / (source_hash + '.py')).read_bytes() == source.encode()
print('Recovered worker:', source_hash)
checkpoint = next(a for a in summary['artifacts'] if a['name'] == 'checkpoint_step_060.pt')
checkpoint


## Next experiment

Follow `data/phase7/NEXT_EXPERIMENT.md`. Verify replay, independently bypass each surviving original block with A restored, and benchmark before accepting further pruning. Numeric replacement tolerances and a fresh blinded image rule must be registered before training. Architecture acceptance precedes step distillation.

## Recovered full-weight healing and image QA

These are historical saved summaries, not newly executed training. Review all candidate checkpoints rather than interpreting a small velocity gain as image-quality promotion.

In [ ]:
def saved_json(relative):
    return json.loads((root / relative).read_bytes())

full_weight = [saved_json('data/phase6/' + name + '/summary.json')
               for name in ('full_student_healing', 'full_student_healing_lr_low_v2')]
[{key: report.get(key) for key in ('status', 'optimizer_updates', 'trainable_parameters', 'selected_step')}
 for report in full_weight]


In [ ]:
decision = published['phase6_checkpoint_decision']
{'evidence': decision['evidence'], 'rule_checks': decision['rule_checks'], 'decision': decision['decision']}


## Historical Phase 7 experiments

The four-block failure and provisional two-block candidate are separate experiments. The adaptive protocol below remains an immutable historical record; new execution is stored under separate request hashes.

In [ ]:
{'four_blocks': published['phase7_four_blocks'],
 'healed_28_blocks': published['phase7_healing_28_blocks'],
 'blocks_4_5': published['phase7_blocks_4_5'],
 'adaptive_protocol': published['phase7_adaptive_pruning_plan']}


## New conditional importance scan

The launcher below performs model computation only when explicitly run. Completed reports are reused; a started run is never silently duplicated. The frozen student is restored by reinstating its unchanged block objects, with parameter identity/version audits and a fresh KV cache for every prompt/trial.

Launch from the repository root:
```powershell
.venv/Scripts/python.exe tools/run_conditional_scan.py
```

In [ ]:
def experiment_records(name):
    reports = []
    for run in sorted((root / 'data/phase7' / name).glob('*')):
        if not run.is_dir():
            continue
        for filename in ('summary.json', 'progress.json', 'call.json'):
            path = run / filename
            if path.exists():
                reports.append({'run': run.name, 'record': filename, 'data': json.loads(path.read_bytes())})
                break
    return reports

scan_records = experiment_records('conditional_importance')
[{key: item['data'].get(key) for key in ('request_key', 'status', 'candidate_A_replay_max_relative_l2', 'ranking', 'error')}
 for item in scan_records]


## Matched latency and VRAM benchmark

Thirty paired prompts/seeds; fifteen AB and fifteen BA orders shuffled before timing, one warmup after each restoration, synchronized wall time, per-forward CUDA events, separate peak memory reset. CPU restoration and PNG serialization are excluded. This benchmark uses the preregistered MATH attention backend; do not generalize absolute times to a different backend.

```powershell
.venv/Scripts/python.exe tools/run_latency_benchmark.py
```

In [ ]:
benchmark_records = experiment_records('latency_benchmark')
[{key: item['data'].get(key) for key in ('status', 'paired_speedup', 'summary', 'error')}
 for item in benchmark_records]


## Fresh blinded quality gate

The 159-prompt protocol was committed before generation. It uses unused VALIDATION entries beyond the 20k pilot, excludes recorded historical IDs, and compares A to the intact teacher. Image identities are randomized per prompt; scores must be locked before revealing the identity key. A single AI review remains a limitation. Passing the image screen permits research only; it does not retroactively pass the historical velocity gate.

```powershell
.venv/Scripts/python.exe tools/run_blind_quality.py
.venv/Scripts/python.exe tools/prepare_blind_review.py
```

In [ ]:
saved_json('data/phase7/blind_quality_v1/protocol.json')


## Recovery and execution monitoring

```powershell
.venv/Scripts/python.exe tools/experiment_status.py
python tools/verify_recovered_tensors.py
```

Large tensors are under ignored `.archive/`; their verification ledger is versioned. Original source bytes and recovered summaries are in Git. Missing local-only files remain listed explicitly in the published-source inventory.